In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/akashkumarsamantray/food-delivery-germany/food-delivery-market-analysis-germany/data/raw_reviews.csv')
df.shape

(5999, 18)

In [2]:
## Missing Values

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5999 entries, 0 to 5998
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   reviewId       5999 non-null   object
 1   userName       5999 non-null   object
 2   content        5999 non-null   object
 3   score          5999 non-null   int64 
 4   thumbsUpCount  5999 non-null   int64 
 5   at             5999 non-null   object
 6   replyContent   5999 non-null   object
 7   repliedAt      1717 non-null   object
 8   appVersion     5999 non-null   object
 9   app_name       5999 non-null   object
 10  review_year    5999 non-null   int64 
 11  review_month   5999 non-null   int64 
 12  review_day     5999 non-null   object
 13  review_hour    5999 non-null   int64 
 14  sentiment      5999 non-null   object
 15  review_length  5999 non-null   int64 
 16  word_count     5999 non-null   int64 
 17  has_reply      5999 non-null   int64 
dtypes: int64(8), object(10)
memo

In [3]:
df.isnull().sum()

reviewId            0
userName            0
content             0
score               0
thumbsUpCount       0
at                  0
replyContent        0
repliedAt        4282
appVersion          0
app_name            0
review_year         0
review_month        0
review_day          0
review_hour         0
sentiment           0
review_length       0
word_count          0
has_reply           0
dtype: int64

In [4]:
## Removed Unnecessary Column

df = df.drop(columns=['userImage', 'reviewCreatedVersion', 'scraped_at'])

KeyError: "['userImage', 'reviewCreatedVersion', 'scraped_at'] not found in axis"

In [5]:
# Drop the 1 row with missing username
df = df.dropna(subset=['userName'])

# Fill missing reply data (most apps didn't reply)
df['replyContent'] = df['replyContent'].fillna('No Reply')
df['repliedAt'] = df['repliedAt'].fillna('No Reply')

# Fill missing app version
df['appVersion'] = df['appVersion'].fillna('Unknown')

In [6]:
## Converted Date Columns

df['at'] = pd.to_datetime(df['at'])
df['repliedAt'] = pd.to_datetime(df['repliedAt'], errors='coerce')

/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_36256/1036400349.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['repliedAt'] = pd.to_datetime(df['repliedAt'], errors='coerce')


In [7]:
## Duplicates

print(df.duplicated().sum())
print(df.duplicated(subset=['reviewId']).sum())

0
0


In [8]:
df.info()
df['score'].describe()
df['app_name'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5999 entries, 0 to 5998
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   reviewId       5999 non-null   object        
 1   userName       5999 non-null   object        
 2   content        5999 non-null   object        
 3   score          5999 non-null   int64         
 4   thumbsUpCount  5999 non-null   int64         
 5   at             5999 non-null   datetime64[ns]
 6   replyContent   5999 non-null   object        
 7   repliedAt      1717 non-null   datetime64[ns]
 8   appVersion     5999 non-null   object        
 9   app_name       5999 non-null   object        
 10  review_year    5999 non-null   int64         
 11  review_month   5999 non-null   int64         
 12  review_day     5999 non-null   object        
 13  review_hour    5999 non-null   int64         
 14  sentiment      5999 non-null   object        
 15  review_length  5999 n

app_name
Wolt          2000
Lieferando    2000
Uber Eats     1999
Name: count, dtype: int64

In [ ]:
## Using Feature Engineering to convert raw data into machine-readable inputs

In [9]:
## Extracting Date features

df['review_year'] = df['at'].dt.year
df['review_month'] = df['at'].dt.month
df['review_day'] = df['at'].dt.day_name()
df['review_hour'] = df['at'].dt.hour

In [10]:
## Sentiment label from score

def sentiment_label(score):
    if score >= 4:
        return 'Positive'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Negative'

df['sentiment'] = df['score'].apply(sentiment_label)

In [11]:
## Review Length Feature

df['review_length'] = df['content'].apply(len)
df['word_count'] = df['content'].apply(lambda x: len(x.split()))

In [12]:
## Has reply Flag

df['has_reply'] = df['replyContent'].apply(lambda x: 0 if x == 'No Reply' else 1)

In [13]:
## Checking all

df[['app_name', 'score', 'sentiment', 'review_length', 'word_count', 'has_reply']].head()

,app_name,score,sentiment,review_length,word_count,has_reply
0,Wolt,5,Positive,33,6,0
1,Wolt,1,Negative,107,17,0
2,Wolt,1,Negative,409,60,0
3,Wolt,5,Positive,9,2,0
4,Wolt,5,Positive,5,1,0


In [14]:
## Saving Cleaned Dataset

df.to_csv('/Users/akashkumarsamantray/food-delivery-germany/food-delivery-market-analysis-germany/data/raw_reviews.csv', index=False)
print("Saved successfully!")
print(df.shape)

Saved successfully!
(5999, 18)
